[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap09_series_temporais/decomposicao_diagnostico_autoveiculos.ipynb)

# Decomposição e diagnóstico de uma série mensal de autoveículos

Este notebook analisa uma série mensal da base SériesTemporais_Autoveículos. O fluxo inclui inspeção e seleção da série, verificação de continuidade, separação temporal com teste reservado antes das escolhas de modelagem, exploração de componentes e diagnósticos de estabilidade.

**Local pressuposto no repositório:** `capitulos/cap09_series_temporais/decomposicao_diagnostico_autoveiculos.ipynb`. Ajuste o caminho do botão se a pasta definitiva for outra.

## 1. Objetivo e protocolo temporal

A decomposição separa uma série em componentes que podem ser interpretados como tendência, sazonalidade e resíduo, conforme o modelo aditivo ou multiplicativo adotado. A decomposição é uma ferramenta descritiva e depende da definição da série, do período, da frequência e das hipóteses sobre os componentes.

O protocolo deste exercício reserva primeiro o período final como teste. A seleção da série, a definição do período de análise, a escolha de transformações e a exploração da decomposição são feitas usando apenas o período de treinamento. O teste é exibido somente ao final para inspeção da evolução não usada nas decisões anteriores. Não se usa o teste para escolher parâmetros ou alterar o modelo.

In [ ]:
import io
import urllib.request
import unicodedata
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller, kpss

SEMENTE=2026
URL_BASE="https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/S%C3%A9riesTemporais_Autove%C3%ADculos.xlsx"
np.random.seed(SEMENTE)

## 2. Carregamento e inspeção

O notebook carrega a planilha diretamente do repositório e exibe as abas, dimensões e primeiras linhas. A identificação da coluna temporal, da medida e de eventuais categorias é feita após essa inspeção; os nomes não são presumidos.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE,timeout=30) as resposta:
            return pd.ExcelFile(io.BytesIO(resposta.read()))
    except Exception as erro:
        print("Download indisponível:",erro)
        try:
            from google.colab import files
            enviados=files.upload()
            if not enviados: raise RuntimeError("Nenhum arquivo selecionado.")
            return pd.ExcelFile(io.BytesIO(enviados[next(iter(enviados))]))
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha localmente.")

excel=carregar_excel()
bases={aba:pd.read_excel(excel,sheet_name=aba) for aba in excel.sheet_names}
for aba,df in bases.items():
    print(f"\n{aba}: {df.shape}")
    display(df.head())
    print("Colunas:",list(df.columns))

## 3. Seleção justificada da série e do período

Escolha uma única série mensal, por exemplo uma medida de vendas ou produção para uma categoria ou total agregado, conforme a estrutura da base. Configure abaixo os nomes exatos da aba, coluna de data, coluna numérica e, se necessário, coluna e valor de filtro. Justifique a escolha e o intervalo analisado no texto de interpretação.

A função de filtro aceita correspondência parcial sem distinção entre maiúsculas e minúsculas para reduzir problemas com pequenas variações textuais. A série deve representar a mesma unidade e cobertura ao longo do tempo.

In [ ]:
# Preencha com nomes exatos observados na inspeção.
ABA=None
COLUNA_DATA=None
COLUNA_VALOR=None
COLUNA_CATEGORIA=None  # use None se não houver filtro por categoria
FILTRO_CATEGORIA=None  # valor ou trecho de texto; use None para não filtrar
DATA_INICIAL=None      # opcional, formato YYYY-MM-DD
DATA_FINAL=None        # opcional, formato YYYY-MM-DD

if not all([ABA,COLUNA_DATA,COLUNA_VALOR]) or ABA not in bases:
    raise ValueError("Configure ABA, COLUNA_DATA e COLUNA_VALOR após inspecionar a planilha.")
df=bases[ABA].copy()
if COLUNA_CATEGORIA and FILTRO_CATEGORIA:
    if COLUNA_CATEGORIA not in df.columns:
        raise KeyError("Coluna de categoria não encontrada.")
    mascara=df[COLUNA_CATEGORIA].astype(str).str.contains(str(FILTRO_CATEGORIA),case=False,na=False,regex=False)
    df=df.loc[mascara].copy()
for c in [COLUNA_DATA,COLUNA_VALOR]:
    if c not in df.columns: raise KeyError(f"Coluna não encontrada: {c}")
df[COLUNA_DATA]=pd.to_datetime(df[COLUNA_DATA],errors="coerce",dayfirst=True)
df[COLUNA_VALOR]=pd.to_numeric(df[COLUNA_VALOR],errors="coerce")
df=df.dropna(subset=[COLUNA_DATA,COLUNA_VALOR]).copy()
if DATA_INICIAL: df=df[df[COLUNA_DATA]>=pd.Timestamp(DATA_INICIAL)]
if DATA_FINAL: df=df[df[COLUNA_DATA]<=pd.Timestamp(DATA_FINAL)]
print("Registros válidos:",len(df))
print("Intervalo observado:",df[COLUNA_DATA].min(),"a",df[COLUNA_DATA].max())
display(df.head())

## 4. Frequência mensal, continuidade e duplicidades

As datas são convertidas em períodos mensais. Antes de decompor, verifica-se se há mais de um registro para a mesma série e mês, se existem meses ausentes e se há valores negativos ou ausentes. Se houver mais de um registro por mês, a agregação precisa corresponder à natureza da medida: soma para fluxos mensais em unidades compatíveis, média para taxas ou outra regra documentada. Não se agrega automaticamente sem essa justificativa.

In [ ]:
df["mes"]=df[COLUNA_DATA].dt.to_period("M")
contagem_mes=df.groupby("mes")[COLUNA_VALOR].size()
print("Meses com mais de um registro:",int((contagem_mes>1).sum()))
print("Maior número de registros em um mês:",int(contagem_mes.max()))

# Regra de agregação configurável e explícita.
AGREGACAO="sum"  # escolher e justificar: 'sum' ou 'mean'
if AGREGACAO not in ["sum","mean"]: raise ValueError("AGREGACAO deve ser 'sum' ou 'mean'.")
serie=df.groupby("mes")[COLUNA_VALOR].agg(AGREGACAO).sort_index()
indice=pd.period_range(serie.index.min(),serie.index.max(),freq="M")
serie=serie.reindex(indice)
serie.index=serie.index.to_timestamp(how="start")
print("Meses no intervalo:",len(serie))
print("Meses ausentes:",int(serie.isna().sum()))
print("Valores negativos:",int((serie.dropna()<0).sum()))
display(serie[serie.isna()].to_frame("valor ausente").head(20))
if serie.isna().any():
    raise ValueError("Há meses ausentes. Investigue a cobertura e documente uma regra antes de decompor; não interpole automaticamente.")
if len(serie)<36:
    raise ValueError("Há menos de 36 observações mensais. Revise o período; a análise sazonal anual requer ciclos suficientes para interpretação.")
if (serie<=0).any():
    print("A série contém valores não positivos; decomposição multiplicativa e logaritmo não são apropriados sem tratamento justificado.")

## 5. Reserva prévia do período de teste

O período final é separado antes das decisões de decomposição. A proporção de teste deve ser escolhida conforme a extensão disponível, mantendo observações suficientes no treinamento para examinar ciclos sazonais. Por padrão, reserva-se aproximadamente os últimos 20% dos meses, com pelo menos 12 meses. Ajuste a proporção apenas com justificativa, sem examinar o desempenho no teste.

In [ ]:
FRACAO_TESTE=0.20
n_teste=max(12,int(np.ceil(len(serie)*FRACAO_TESTE)))
if len(serie)-n_teste<24:
    raise ValueError("O treinamento ficaria com menos de 24 meses. Ajuste o período ou a fração com justificativa.")
treino=serie.iloc[:-n_teste].copy()
teste=serie.iloc[-n_teste:].copy()
print("Treinamento:",treino.index.min().date(),"a",treino.index.max().date(),"| n=",len(treino))
print("Teste reservado:",teste.index.min().date(),"a",teste.index.max().date(),"| n=",len(teste))
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(treino,label="Treinamento")
ax.axvspan(teste.index.min(),teste.index.max(),color="orange",alpha=.15,label="Teste reservado")
ax.set_title("Série mensal e corte temporal definido antes da modelagem")
ax.set_ylabel(COLUNA_VALOR);ax.legend();plt.show()

## 6. Exploração no treinamento e notas metodológicas

A tendência visual e os padrões recorrentes são examinados somente no treinamento. A decomposição aditiva representa a série como soma de componentes; a multiplicativa expressa componentes relativos e exige valores positivos. A sazonalidade anual é avaliada com período 12, desde que a frequência mensal esteja correta e haja ciclos suficientes.

A decomposição clássica utiliza médias móveis e pode perder observações nas extremidades. STL estima tendência e sazonalidade por suavização local e pode ser mais flexível, mas sua configuração também influencia o resultado. Componentes decompostos são descrições dependentes do método, não entidades observadas diretamente.

In [ ]:
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(treino,color="steelblue")
ax.set_title("Série mensal no período de treinamento")
ax.set_ylabel(COLUNA_VALOR);plt.show()

# Decomposição aditiva clássica para periodicidade anual, apenas no treinamento.
resultado_aditivo=seasonal_decompose(treino,model="additive",period=12,extrapolate_trend="freq")
resultado_aditivo.plot();plt.suptitle("Decomposição aditiva clássica (treinamento)",y=1.02);plt.show()

# STL como alternativa robusta a variações locais, sem uso do teste.
resultado_stl=STL(treino,period=12,robust=True).fit()
resultado_stl.plot();plt.suptitle("Decomposição STL robusta (treinamento)",y=1.02);plt.show()

if (treino>0).all():
    resultado_mult=seasonal_decompose(treino,model="multiplicative",period=12,extrapolate_trend="freq")
    resultado_mult.plot();plt.suptitle("Decomposição multiplicativa clássica (treinamento)",y=1.02);plt.show()
else:
    print("Decomposição multiplicativa omitida: existem valores não positivos no treinamento.")

## 7. Diagnóstico descritivo dos componentes e estabilidade

Examine a tendência, o padrão sazonal e o resíduo. Avalie se a amplitude sazonal parece aproximadamente constante (compatível com descrição aditiva) ou se varia proporcionalmente ao nível (situação em que uma representação multiplicativa ou transformação pode ser considerada, se os dados permitirem). Compare os padrões entre subperíodos do treinamento para verificar se a sazonalidade e o nível parecem estáveis.

Os testes ADF e KPSS são apresentados como diagnósticos complementares de estacionariedade, com hipóteses nulas distintas. Seus resultados não devem ser interpretados isoladamente nem como prova definitiva, especialmente diante de quebras, sazonalidade, tendências determinísticas ou amostras curtas.

In [ ]:
# Médias sazonais por mês do calendário em metades do treinamento.
partes=np.array_split(treino,2)
perfis=[]
for i,parte in enumerate(partes,1):
    perfis.append(parte.groupby(parte.index.month).mean().rename(f"Subperíodo {i}"))
perfis_mes=pd.concat(perfis,axis=1)
perfis_mes.index.name="Mês do ano"
display(perfis_mes.round(2))
perfis_mes.plot(marker="o",figsize=(10,4),title="Médias por mês do calendário em subperíodos do treinamento")
plt.xlabel("Mês");plt.ylabel(COLUNA_VALOR);plt.show()

# Testes de raiz unitária/estacionariedade na série de treinamento.
print("ADF (H0: presença de raiz unitária):")
adf=adfuller(treino,autolag="AIC")
print({"estatística":adf[0],"p-valor":adf[1],"defasagens":adf[2],"n":adf[3]})
print("\nKPSS (H0: estacionariedade em nível):")
try:
    kp=kpss(treino,regression="c",nlags="auto")
    print({"estatística":kp[0],"p-valor":kp[1],"defasagens":kp[2]})
except Exception as erro:
    print("KPSS não calculado:",erro)

# ACF dos resíduos é opcional e apenas exploratória.
from statsmodels.graphics.tsaplots import plot_acf
residuos=resultado_stl.resid.dropna()
plot_acf(residuos,lags=min(24,len(residuos)//2-1),zero=False)
plt.title("Autocorrelação dos resíduos STL no treinamento");plt.show()

## 8. Inspeção final do período de teste

Somente após concluir as escolhas e diagnósticos do treinamento, o período de teste é mostrado em relação à continuação da série. Esta comparação é visual e não constitui uma avaliação preditiva formal, pois o exercício não ajustou um modelo de previsão. O teste não deve ser incorporado retroativamente às escolhas de série, transformação, período sazonal ou método de decomposição.

In [ ]:
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(treino,label="Treinamento")
ax.plot(teste,label="Teste reservado",color="darkorange")
ax.axvline(teste.index.min(),color="black",linestyle="--",alpha=.7)
ax.set_title("Inspeção final da série: treinamento e teste")
ax.set_ylabel(COLUNA_VALOR);ax.legend();plt.show()
print("Teste reservado, resumo descritivo:")
display(teste.describe().to_frame("estatística").round(2))

## 9. Síntese e checklist

A decomposição e os testes fornecem evidências descritivas sobre tendência, sazonalidade, resíduos e possíveis mudanças no comportamento da série. A conclusão deve registrar a série e o período selecionados, a frequência, a regra de agregação, os meses ausentes ou duplicados, o método de decomposição e as limitações dos diagnósticos.

### Checklist

- A série representa uma medida e uma cobertura consistentes ao longo do período?
- A agregação mensal está de acordo com a natureza da variável?
- A continuidade foi verificada e meses ausentes foram investigados?
- O período de teste foi reservado antes das escolhas de modelagem?
- Tendência e sazonalidade foram exploradas apenas no treinamento?
- As conclusões de estacionariedade consideram as hipóteses diferentes de ADF e KPSS?
- Os componentes foram descritos como estimativas dependentes do método?
- A estabilidade foi examinada entre subperíodos sem consultar o teste?
- As notas metodológicas e limitações foram registradas?